In [50]:
import torch
words=open('names.txt','r').read().splitlines()

In [51]:
stoi={'.':0}
stoi={ch:i+1 for i,ch in enumerate(list("abcdefghijklmnopqrstuvwxyz"))}
stoi['.']=0
itos={i:ch for ch,i in stoi.items()}

In [52]:
n=5

def MakeDataSet(wo):
    context=[0]*n
    x=[]
    y=[]
    for w in wo:
        for ch in list(w)+['.']:
            x.append(context)
            y.append(stoi[ch])
            context=context[1:]+[stoi[ch]]

    return torch.tensor(x),torch.tensor(y)

import random
random.shuffle(words)
n1=int(0.9*len(words))

Xtr,Ytr=MakeDataSet(words[:n1])
Xte,Yte=MakeDataSet(words[n1:])

In [53]:
d=6
c=torch.randn(27,d).float()
W1=torch.randn(d*n,100)*0.2
B1=torch.randn(100)*0.1
W2=torch.randn(100,27)*0.01
B2=torch.randn(27)*0.0
p=[c,W1,W2,B1,B2]

In [54]:
for a in p:
    a.requires_grad=True

In [55]:
for i in range(200000):
    ix = torch.randint(0, Xtr.shape[0], (32,))

    emb=c[Xtr[ix]]
    h = torch.tanh(emb.view(-1, d*n) @ W1 + B1)
    logits = h @ W2 + B2
    loss = torch.nn.functional.cross_entropy(logits, Ytr[ix])
    loss += 0.01 * ((W1).mean() + (W2).mean())

    for a in p:
        a.grad = None
    loss.backward()

    lr = 0.1 if i < 100000 else 0.01
    for a in p:
        a.data += -lr * a.grad

print(loss.item())

1.9426624774932861


In [56]:
g = torch.Generator().manual_seed(123456789)

for _ in range(20):
    
    out = []
    context = [0] * n
    while True:
      emb = c[torch.tensor([context])]
      h = torch.tanh(emb.view(1, -1) @ W1 + B1)
      logits = h @ W2 + B2
      probs = torch.nn.functional.softmax(logits, dim=1)
      ix = torch.multinomial(probs, num_samples=1, generator=g).item()
      context = context[1:] + [ix]
      out.append(ix)
      if ix == 0:
        break
    
    print(''.join(itos[i] for i in out))

rexley.
rislith.
rnkie.
rwolyn.
mamila.
mynalis.
zesan.
pitaliollah.
rusyn.
rwala.
ramilonar.
dzy.
cristin.
rpeeth.
dwa.
osmite.
amrie.
cesri.
lenntri.
runnah.


In [57]:
with torch.no_grad():
    emb = c[Xte]
    h = torch.tanh(emb.view(-1, d*n) @ W1 + B1)
    logits = h @ W2 + B2
    val_loss = torch.nn.functional.cross_entropy(logits, Yte)

print(f"Final Training Loss:   {loss.item():.4f}")
print(f"Final Validation Loss: {val_loss.item():.4f}")

Final Training Loss:   1.9427
Final Validation Loss: 2.1098
